# 🧠 03 — 5-Fold CV Training with Dynamic Undersampling
**Project:** ISIC 2024 — Skin Cancer Detection

**Goal:** Train EfficientNetV2-S across 5 Patient-level Group K-Folds using our Dynamic Undersampling Schedule — inspired by 1st/2nd place approaches but with a distinct negative:positive ratio warmup.

---

## 📦 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import albumentations as A
import cv2
from pathlib import Path
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
IMG_SIZE    = 128
BATCH_SIZE  = 64
N_FOLDS     = 5
EPOCHS      = 30
LR          = 3e-4
SEED        = 42
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_DIR = Path('../data/processed')
IMG_DIR  = Path('../data/raw/train-image/image')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

# Dynamic undersampling schedule: epoch → negative:positive ratio
# Inspired by 2nd place (fixed 1:3), but warmed up from 1:20 → 1:5 → 1:3
UNDERSAMPLE_SCHEDULE = {
    **{e: 20 for e in range(1, 6)},    # Epochs 1-5:  ratio 1:20
    **{e: 5  for e in range(6, 16)},   # Epochs 6-15: ratio 1:5
    **{e: 3  for e in range(16, 100)}, # Epochs 16+:  ratio 1:3
}

print(f'Device: {DEVICE}')
print(f'Undersampling schedule (epoch→neg:pos): {list(dict.fromkeys(UNDERSAMPLE_SCHEDULE.values()))}')

## 📂 2. Dataset with Dynamic Undersampling

In [ ]:
class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None):
        self.df         = df.reset_index(drop=True)
        self.img_dir    = Path(img_dir)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = str(self.img_dir / f"{row['isic_id']}.jpg")
        img  = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)
        if self.transforms:
            img = self.transforms(image=img)['image']
        img = torch.tensor(img.transpose(2, 0, 1), dtype=torch.float32)
        label = torch.tensor(row['target'], dtype=torch.float32)
        return img, label


def get_sampler(df, neg_pos_ratio: int):
    """Create WeightedRandomSampler to achieve the given neg:pos ratio."""
    n_pos = df['target'].sum()
    n_neg = len(df) - n_pos
    # Weight malignant samples higher
    weights = np.where(df['target'].values == 1,
                       n_neg / (n_pos * neg_pos_ratio),  # malignant weight
                       1.0)                               # benign weight (baseline)
    n_samples = int(n_pos * (neg_pos_ratio + 1))         # total samples per epoch
    return WeightedRandomSampler(weights=weights, num_samples=n_samples, replacement=True)


train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.Rotate(limit=45, p=0.7),
    A.RandomBrightnessContrast(p=0.6),
    A.HueSaturationValue(p=0.4),
    A.CoarseDropout(max_holes=6, max_height=16, max_width=16, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

print('Dataset classes and augmentation pipeline defined.')

## 🏗️ 3. Model: EfficientNetV2-S (timm)

In [ ]:
class SkinCancerModel(nn.Module):
    def __init__(self, model_name='efficientnetv2_s', pretrained=True, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features).squeeze(1)


class FocalLoss(nn.Module):
    """Binary Focal Loss — reduces loss contribution from easy benign examples."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce   = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt    = torch.where(targets == 1, probs, 1 - probs)
        at    = torch.where(targets == 1, torch.tensor(self.alpha), torch.tensor(1 - self.alpha))
        loss  = at * (1 - pt) ** self.gamma * bce
        return loss.mean()


# Quick sanity check
model_test = SkinCancerModel().to(DEVICE)
dummy      = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out        = model_test(dummy)
print(f'Model output shape: {out.shape}  (should be torch.Size([2]))')
n_params   = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')
del model_test, dummy

## 🚀 4. 5-Fold CV Training Loop with Dynamic Undersampling

In [ ]:
df = pd.read_csv(DATA_DIR / 'train_folds.csv')

oof_preds = np.zeros(len(df))
fold_aucs = []

for fold in range(N_FOLDS):
    print(f'\n{"="*55}')
    print(f'  FOLD {fold+1}/{N_FOLDS}')
    print(f'{"="*55}')

    train_df = df[df['fold'] != fold].reset_index(drop=True)
    val_df   = df[df['fold'] == fold].reset_index(drop=True)

    val_ds     = ISICDataset(val_df, IMG_DIR, val_transforms)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

    model     = SkinCancerModel().to(DEVICE)
    criterion = FocalLoss(gamma=2.0)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

    best_auc   = 0
    train_losses = []
    val_aucs     = []

    for epoch in range(1, EPOCHS + 1):
        # Dynamic undersampling — ratio changes per epoch
        neg_pos_ratio = UNDERSAMPLE_SCHEDULE.get(epoch, 3)
        sampler    = get_sampler(train_df, neg_pos_ratio)
        train_ds   = ISICDataset(train_df, IMG_DIR, train_transforms)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                  num_workers=2, pin_memory=True, drop_last=True)

        # ── Train ──
        model.train()
        epoch_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            if scaler:
                with torch.cuda.amp.autocast():
                    logits = model(imgs)
                    loss   = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(imgs)
                loss   = criterion(logits, labels)
                loss.backward()
                optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)

        # ── Validate ──
        model.eval()
        val_probs = []
        with torch.no_grad():
            for imgs, _ in val_loader:
                logits = model(imgs.to(DEVICE))
                val_probs.extend(torch.sigmoid(logits).cpu().numpy())

        val_auc = roc_auc_score(val_df['target'].values, val_probs)
        val_aucs.append(val_auc)

        print(f'Epoch {epoch:02d} | ratio=1:{neg_pos_ratio} | Loss={avg_loss:.4f} | Val AUC={val_auc:.4f}')

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), MODEL_DIR / f'fold{fold}_best.pt')
            # Save OOF predictions
            val_indices = df[df['fold'] == fold].index
            oof_preds[val_indices] = val_probs

    fold_aucs.append(best_auc)
    print(f'\nFold {fold+1} Best AUC: {best_auc:.4f}')

print(f'\n{"="*55}')
print(f'CV Results: {[round(a,4) for a in fold_aucs]}')
print(f'Mean OOF AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')
print(f'{"="*55}')

# Save OOF predictions for GBDT stacking
df['oof_pred_effnet'] = oof_preds
df.to_csv('../results/oof_predictions.csv', index=False)
print('OOF predictions saved.')

## 📈 5. Training Curves

In [ ]:
# (run after training completes)
# Plot fold AUC comparison
plt.figure(figsize=(10, 5))
plt.bar([f'Fold {i+1}' for i in range(N_FOLDS)], fold_aucs,
        color=plt.cm.Blues(np.linspace(0.5, 0.9, N_FOLDS)), edgecolor='white', linewidth=1.5)
plt.axhline(np.mean(fold_aucs), color='red', linestyle='--', linewidth=2,
            label=f'Mean AUC = {np.mean(fold_aucs):.4f}')
for i, auc in enumerate(fold_aucs):
    plt.text(i, auc + 0.001, f'{auc:.4f}', ha='center', va='bottom', fontweight='bold')
plt.title('Validation AUC per Fold — 5-Fold Patient GroupKFold', fontsize=13, fontweight='bold')
plt.ylabel('AUC Score'); plt.ylim(0.7, 1.0); plt.legend()
plt.tight_layout()
plt.savefig('../results/figures/fold_auc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## ✅ 6. Summary

| Fold | Best AUC |
|------|----------|
| 1 | *(fill after training)* |
| 2 | *(fill after training)* |
| 3 | *(fill after training)* |
| 4 | *(fill after training)* |
| 5 | *(fill after training)* |
| **Mean ± Std** | *(fill after training)* |

OOF predictions saved to `results/oof_predictions.csv`.

**Next:** `04_Evaluation_Submission.ipynb` — compute pAUC, train GBDT meta-learner, TTA, generate submission.